# Swin Semantic Segmentation — DIMER TASK-INFERENCE tutorial

**Profile:** `TASK-INFERENCE`  
**Notebook spec:** DIMER Notebook Specification 1.0  
**Capability:** pretrained semantic segmentation with the repository's public `DimerSwinSegmenter` API.

This tutorial runs the pinned **Swin-T + UPerNet** ADE20K model through the repository-owned task-inference runtime. **No fine-tuning occurs.** The model maps an RGB image to one ADE20K semantic class index per pixel.

By the end you will be able to bootstrap the exact runtime, verify the checkpoint before loading it, validate images, run semantic segmentation, evaluate a fixed labelled sample with mIoU and per-class IoU, optionally segment your own image, and export semantic masks, metrics, per-class results, and provenance.

The default sample consists of two immutable ADE20K validation fixtures (`ADE_val_00000001` and `ADE_val_00000002`) published by Hugging Face's internal testing dataset. These are real labelled ADE20K examples, but two images are only **tutorial sanity evidence** and do not reproduce the upstream ADE20K benchmark.


## Prerequisites and trust boundary

- **Supported runtime:** CPython 3.10 Jupyter on Linux; CPU is the default path. GPU is not required.
- **Network:** GitHub, PyPI/OpenMMLab package indexes, the OpenMMLab checkpoint host, and Hugging Face for the fixed ADE20K fixtures.
- **Model serialization:** the upstream `.pth` file is code-capable PyTorch serialization. The repository verifies its exact byte size and SHA-256 **before** MMSegmentation loads it. A matching digest establishes byte identity, not sender authenticity; use only the pinned trusted upstream source.
- **BYOD:** optional and disabled by default. Do not upload confidential or restricted imagery to a notebook environment you are not authorized to use. Images remain inside the notebook runtime and are not sent to an inference service.

Current OpenMMLab wheels for this frozen stack require Python 3.10 and NumPy 1.x. The notebook fails clearly rather than silently changing the model stack.


## 1. Bootstrap the immutable repository runtime

This cell checks out the exact repository revision that implements `DimerSwinSegmenter`, installs one pinned CPU dependency graph, installs the repository package without resolving another graph, and prints the effective versions. **Look for:** Python 3.10, torch 2.1.2, MMSegmentation 1.2.2, MMCV 2.1.0, MMEngine 0.10.7, NumPy 1.26.4.


In [ ]:
from __future__ import annotations
import csv, hashlib, json, os, subprocess, sys, urllib.request
from pathlib import Path

if sys.version_info[:2] != (3, 10):
    raise RuntimeError(f'Python 3.10 is required by the qualified OpenMMLab runtime; current interpreter is {sys.version.split()[0]}. Use a Python 3.10 Jupyter kernel.')

REPOSITORY = 'https://github.com/kurtvalcorza/swin-segmentation-pipeline.git'
REPOSITORY_REF = '1a9440d42f8e21f3a9956221aa9e09857f9a10c1'
WORK = Path(os.environ.get('DIMER_TUTORIAL_WORKSPACE', '/tmp/dimer-swin-segmentation-tutorial')).resolve()
REPO = WORK / 'repo'
SAMPLE = WORK / 'sample'
OUTPUTS = WORK / 'outputs'
for p in (WORK, SAMPLE, OUTPUTS): p.mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=None):
    cmd=[str(x) for x in cmd]; print('+',' '.join(cmd)); subprocess.run(cmd,cwd=cwd,check=True)

if not REPO.exists(): run(['git','clone','--filter=blob:none',REPOSITORY,REPO])
run(['git','checkout','--detach',REPOSITORY_REF],cwd=REPO)
head=subprocess.run(['git','rev-parse','HEAD'],cwd=REPO,check=True,capture_output=True,text=True).stdout.strip()
assert head == REPOSITORY_REF
run([sys.executable,'-m','pip','install','--disable-pip-version-check','--index-url','https://download.pytorch.org/whl/cpu','torch==2.1.2','torchvision==0.16.2'])
run([sys.executable,'-m','pip','install','--disable-pip-version-check','openmim==0.3.9'])
run(['mim','install','mmengine==0.10.7'])
run(['mim','install','mmcv==2.1.0'])
run([sys.executable,'-m','pip','install','--disable-pip-version-check','mmsegmentation==1.2.2','ftfy==6.3.1','regex==2024.11.6'])
run([sys.executable,'-m','pip','install','--disable-pip-version-check','--force-reinstall','numpy==1.26.4','opencv-python==4.10.0.84'])
run([sys.executable,'-m','pip','install','--disable-pip-version-check','--no-deps',REPO])
run([sys.executable,'-m','pip','check'])

import importlib.metadata as md, numpy as np, torch
runtime={'python':sys.version.split()[0],'torch':torch.__version__,'numpy':np.__version__,'mmsegmentation':md.version('mmsegmentation'),'mmcv':md.version('mmcv'),'mmengine':md.version('mmengine'),'repositoryRevision':REPOSITORY_REF}
print(json.dumps(runtime,indent=2))


## 2. Acquire and validate the labelled ADE20K tutorial sample

The four sample files are fetched from an **immutable dataset commit** (`850d349…`): two JPEG images and their two PNG semantic maps. No split is invented or changed. ADE20K's raw masks encode `0` as ignore/unlabelled and semantic classes as `1..150`; the MMSegmentation configuration uses `reduce_zero_label=True`, so evaluation maps raw labels `1..150` to model class indices `0..149` and excludes raw zero pixels. File SHA-256 values are recorded in provenance.


In [ ]:
HF_COMMIT='850d349e5038f291284e7999fcacbedc0922534b'
HF_BASE=f'https://huggingface.co/datasets/hf-internal-testing/fixtures_ade20k/resolve/{HF_COMMIT}'
sample_pairs=[]; sample_digests={}
for stem in ('ADE_val_00000001','ADE_val_00000002'):
    image=SAMPLE/f'{stem}.jpg'; mask=SAMPLE/f'{stem}.png'
    for target in (image,mask):
        if not target.exists(): urllib.request.urlretrieve(f'{HF_BASE}/{target.name}',target)
        sample_digests[target.name]=hashlib.sha256(target.read_bytes()).hexdigest()
    sample_pairs.append((image,mask))
from dimer_swin_segmentation.runtime import validate_image
from PIL import Image
sample_summary=[]
for image,mask in sample_pairs:
    info=validate_image(image)
    with Image.open(mask) as m:
        raw=np.array(m)
    if raw.shape != (info['height'],info['width']): raise RuntimeError(f'Image/mask shape mismatch for {image.name}: image {info}, mask {raw.shape}')
    if raw.min() < 0 or raw.max() > 150: raise RuntimeError(f'ADE20K fixture labels outside expected raw range 0..150: {raw.min()}..{raw.max()}')
    sample_summary.append({'image':image.name,'mask':mask.name,'width':info['width'],'height':info['height'],'rawLabelMin':int(raw.min()),'rawLabelMax':int(raw.max())})
print(json.dumps({'sampleType':'public ADE20K validation fixtures','sourceCommit':HF_COMMIT,'files':sample_summary,'sha256':sample_digests},indent=2))


## 3. Resolve the pinned model and run the real repository API

`DimerSwinSegmenter` downloads the exact OpenMMLab checkpoint, checks its expected 240,154,742-byte size and SHA-256 `e380ad3e…b6064`, then passes it to MMSegmentation. The repository API returns a 2-D uint8 semantic mask whose values are ADE20K model class indices `0..149`; it does not expose calibrated per-pixel uncertainty.


In [ ]:
from dimer_swin_segmentation import DimerSwinSegmenter, MODEL_SPEC
segmenter=DimerSwinSegmenter(cache_dir=WORK/'models',device='cpu')
results=[]; prediction_summaries=[]
for image,_ in sample_pairs:
    result=segmenter.predict(image); results.append(result)
    mask_path=OUTPUTS/f'{image.stem}.semantic.png'; result.save_mask(mask_path)
    summary=result.summary(); summary['mask_file']=mask_path.name; prediction_summaries.append(summary)
(OUTPUTS/'predictions.json').write_text(json.dumps(prediction_summaries,indent=2)+'\n')
print(json.dumps({'model':MODEL_SPEC['runtime_id'],'predictions':prediction_summaries},indent=2))


## 4. Evaluate mIoU and per-class IoU on the frozen sample

For each image, raw ADE20K label `0` is ignored and labels `1..150` are shifted to model indices `0..149`. We aggregate pixel intersections and unions across both fixed images, then compute IoU per class and mean IoU over classes present in the sample. Pixel accuracy is reported as a complementary measure. A constant-majority-pixel predictor is included as a trivial sample baseline; because the majority class is derived from this same tiny evaluation sample, it is only a descriptive reference, not an independent benchmark.


In [ ]:
NUM_CLASSES=MODEL_SPEC['num_classes']; IGNORE=255
intersections=np.zeros(NUM_CLASSES,dtype=np.int64); unions=np.zeros(NUM_CLASSES,dtype=np.int64)
correct=0; valid_pixels=0; gt_counts=np.zeros(NUM_CLASSES,dtype=np.int64)
for result,(_,mask_path) in zip(results,sample_pairs):
    raw=np.array(Image.open(mask_path),dtype=np.uint16)
    gt=np.full(raw.shape,IGNORE,dtype=np.uint16); valid=raw>0; gt[valid]=raw[valid]-1
    pred=result.mask.astype(np.uint16,copy=False)
    if pred.shape != gt.shape: raise RuntimeError(f'Prediction/ground-truth shape mismatch: {pred.shape} vs {gt.shape}')
    correct += int(((pred==gt)&valid).sum()); valid_pixels += int(valid.sum())
    gt_counts += np.bincount(gt[valid].astype(np.int64),minlength=NUM_CLASSES)[:NUM_CLASSES]
    for c in range(NUM_CLASSES):
        p=(pred==c)&valid; g=(gt==c)&valid
        intersections[c] += int((p&g).sum()); unions[c] += int((p|g).sum())
ious=np.divide(intersections,unions,out=np.full(NUM_CLASSES,np.nan,dtype=float),where=unions>0)
present=np.flatnonzero(unions>0); miou=float(np.nanmean(ious[present])); pixel_accuracy=float(correct/valid_pixels)
majority_class=int(gt_counts.argmax()); baseline_inter=np.zeros(NUM_CLASSES,dtype=np.int64); baseline_union=np.zeros(NUM_CLASSES,dtype=np.int64)
for _,(_,mask_path) in zip(results,sample_pairs):
    raw=np.array(Image.open(mask_path),dtype=np.uint16); valid=raw>0; gt=np.full(raw.shape,IGNORE,dtype=np.uint16); gt[valid]=raw[valid]-1; baseline=np.full(raw.shape,majority_class,dtype=np.uint16)
    for c in range(NUM_CLASSES):
        p=(baseline==c)&valid; g=(gt==c)&valid; baseline_inter[c]+=int((p&g).sum()); baseline_union[c]+=int((p|g).sum())
baseline_ious=np.divide(baseline_inter,baseline_union,out=np.full(NUM_CLASSES,np.nan,dtype=float),where=baseline_union>0); baseline_miou=float(np.nanmean(baseline_ious[np.flatnonzero(baseline_union>0)]))
rows=[]
for c in present:
    rows.append({'class_id':int(c),'class_name':segmenter.classes[c],'intersection_pixels':int(intersections[c]),'union_pixels':int(unions[c]),'iou':float(ious[c])})
with (OUTPUTS/'per-class-iou.csv').open('w',newline='') as f:
    writer=csv.DictWriter(f,fieldnames=['class_id','class_name','intersection_pixels','union_pixels','iou']); writer.writeheader(); writer.writerows(rows)
metrics={'estimation':'two fixed ADE20K validation fixtures; aggregate pixel IoU','num_images':len(sample_pairs),'valid_pixels':valid_pixels,'classes_with_union':len(present),'miou':miou,'pixel_accuracy':pixel_accuracy,'sample_majority_class_id':majority_class,'sample_majority_class_name':segmenter.classes[majority_class],'sample_majority_constant_baseline_miou':baseline_miou,'upstream_full_ade20k_miou_not_measured_here':MODEL_SPEC['upstream_reported_miou']}
(OUTPUTS/'metrics.json').write_text(json.dumps(metrics,indent=2)+'\n'); print(json.dumps(metrics,indent=2)); print(rows[:10])


## 5. Optional BYOD new-image inference

This branch is disabled by default so the sample path never blocks on an upload dialog. Set `USE_BYOD=True` and provide `BYOD_PATH`; in Colab, leaving the path empty opens the upload picker. The repository validates readability and a 64-megapixel ceiling before model execution. BYOD has no mIoU without a corresponding labelled semantic map; this section is inference only.


In [ ]:
USE_BYOD=False  # @param {type:'boolean'}
BYOD_PATH=''   # @param {type:'string'}
if USE_BYOD:
    p=Path(BYOD_PATH).expanduser() if BYOD_PATH else None
    if p is None:
        try:
            from google.colab import files
            uploaded=files.upload(); name=next(iter(uploaded)); p=WORK/name; p.write_bytes(uploaded[name])
        except Exception as exc:
            raise RuntimeError('Set BYOD_PATH to a local image, or use Colab upload.') from exc
    from dimer_swin_segmentation.runtime import validate_image
    print(validate_image(p)); byod=segmenter.predict(p); out=OUTPUTS/f'{p.stem}.byod-semantic.png'; byod.save_mask(out); print(json.dumps({**byod.summary(),'mask_file':out.name},indent=2))
else:
    print('BYOD skipped (default).')


## 6. Export provenance and verify outputs

The provenance record binds the effective repository runtime, OpenMMLab versions, model identifier, checkpoint digest, the 150-class ordering, fixed sample file identities, and tutorial metrics. It contains no credentials. Semantic masks are saved as PNG class-index rasters, while metrics and per-class IoU remain machine-readable independently of notebook state.


In [ ]:
provenance=segmenter.provenance(); provenance['tutorial']={'notebookProfile':'TASK-INFERENCE','notebookSpec':'1.0','repositoryRevision':REPOSITORY_REF,'sampleSourceCommit':HF_COMMIT,'sampleFilesSha256':sample_digests,'metrics':metrics}
(OUTPUTS/'provenance.json').write_text(json.dumps(provenance,indent=2)+'\n')
required=['predictions.json','metrics.json','per-class-iou.csv','provenance.json']+[f'{image.stem}.semantic.png' for image,_ in sample_pairs]
missing=[name for name in required if not (OUTPUTS/name).is_file()]
if missing: raise RuntimeError(f'Missing expected outputs: {missing}')
print(json.dumps({'outputs':[str(OUTPUTS/n) for n in required],'checkpointVerified':provenance['effective']['checkpoint_sha256']==MODEL_SPEC['checkpoint_sha256']},indent=2))


## Interpretation, limits, and next experiments

A successful run proves that this repository revision can acquire the pinned pretrained Swin-T + UPerNet bytes, verify them, reconstruct the official MMSegmentation inference path, validate and segment new RGB images, compute task-appropriate mIoU/per-class IoU on labelled data, and export portable class-index masks and provenance. It does **not** prove reproduction of the full ADE20K benchmark, robustness to your domain, calibrated per-pixel uncertainty, fairness, safety, production fitness, or support for training/fine-tuning.

The displayed two-image mIoU is deliberately a small-sample sanity check and has high sampling variance. For a deployment decision, use a representative labelled holdout from the target domain, preserve any spatial/group boundaries needed to avoid leakage, inspect class-specific IoU and confusion patterns, and characterize performance under changes in camera, resolution, lighting, and scene composition.
